# Retrieval Demo Notebook
This notebook is a simple public companion to the standalone Study Genie extract. It shows the retrieval pipeline step by step on the demo corpus already stored in the repository.

For reproducibility, the notebook uses a deterministic mock embedding path by default. The real API-based embedding call remains implemented in `src/study_genie_extract/embeddings.py`.

In [ ]:
from pathlib import Path
import json
import sys

ROOT = Path('..').resolve()
SRC = ROOT / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from study_genie_extract.preprocessing import preprocess_document_text
from study_genie_extract.chunking import chunk_text
from study_genie_extract.pipeline import load_documents, build_index
from study_genie_extract.retrieval import retrieve_top_k

## 1. Load the demo corpus

In [ ]:
documents = load_documents(ROOT / 'data' / 'sample_documents.jsonl')
[(doc.document_id, doc.title, len(doc.text)) for doc in documents]

## 2. Show preprocessing

In [ ]:
raw_text = documents[0].text.replace('', '\r\r')
clean_text = preprocess_document_text(raw_text)
{'raw_preview': raw_text[:90], 'clean_preview': clean_text[:90]}

## 3. Show chunking

In [ ]:
sample_chunks = chunk_text(documents[0].text, max_chunk_size=80, overlap=20, separator='')
[(chunk.chunk_index, chunk.start_index, chunk.end_index, chunk.content) for chunk in sample_chunks]

 4. Build a reproducible mock embedding path
This is only for the notebook. The repository also contains the real API-backed embedding function.

In [ ]:
VOCABULARY = ['derivative', 'cubed', 'gradient', 'photosynthesis', 'sigmoid', 'chlorophyll', 'energy', 'probability']

VOCABULARY = ['derivative', 'cubed', 'gradient', 'photosynthesis', 'sigmoid', 'chlorophyll', 'energy', 'probability']

def mock_embedding(text):
    lowered = text.lower()
    return [1.0 if token in lowered else 0.0 for token in VOCABULARY]

def mock_embed_batch(texts, model, api_key):
    return [mock_embedding(text) for text in texts]

def mock_embed_query(text, model, api_key):
    return mock_embedding(text)

mock_embedding(documents[0].text)

def mock_embed_batch(texts, model, api_key):
    return [mock_embedding(text) for text in texts]

def mock_embed_query(text, model, api_key):
    return mock_embedding(text)

mock_embedding(documents[0].text)

## 5. Build the index and run retrieval

In [ ]:
index = build_index(documents, model='mock-embedding', api_key=None, embed_chunks=mock_embed_batch)
len(index)

In [ ]:
queries = [
    'rule for x cubed derivative',
    'what controls the step size in gradient descent',
    'what maps a score to a probability',
]

results = {}
for query in queries:
    hits = retrieve_top_k(query, index, top_k=3, model='mock-embedding', api_key=None, embed_query=mock_embed_query)
    results[query] = [
        {'document_id': hit.chunk.document_id, 'chunk_index': hit.chunk.chunk_index, 'similarity': round(hit.similarity, 4)}
        for hit in hits
    ]

results

## 6. Error analysis example

The next query is intentionally phrased as a paraphrase. It should point to the biology document, but the mock embedding path is weak on unseen wording.

In [ ]:
error_query = 'how do plants make food from sunlight'
error_hits = retrieve_top_k(error_query, index, top_k=3, model='mock-embedding', api_key=None, embed_query=mock_embed_query)
[{'document_id': hit.chunk.document_id, 'similarity': round(hit.similarity, 4)} for hit in error_hits]

Expected result: `bio_1`

Actual result: the top hit is not `bio_1`.

Diagnosis: with the mock embedding path, the query wording does not overlap with the small keyword vocabulary, so the query vector is too weak.

Attempted fix: rephrase the query using terms closer to the corpus.

In [ ]:
fixed_query = 'chemical energy in plants'
fixed_hits = retrieve_top_k(fixed_query, index, top_k=3, model='mock-embedding', api_key=None, embed_query=mock_embed_query)
[{'document_id': hit.chunk.document_id, 'similarity': round(hit.similarity, 4)} for hit in fixed_hits]